# lemonbeat-python Mower
## Choose Log Level

In [ ]:
import logging

import ipywidgets as widgets

widgets.interact(
    logging.getLogger().setLevel,
    level=["ERROR", "WARNING", "INFO", "DEBUG"], options={"level": {"value": "INFO"}}
).widget.children[0].value = "WARNING"


## Setup Library

This will load the config file at `~/.config/lemonbeat-python/default.toml`. If it does not exist, a random inclusion message will be generated and persisted in a newly created config file.

In order to add devices to the list, either edit the config file manually or run the following code to add a new device (replace name and IPv6 address accordingly):
```python
from lemonbeat import Device, config_file
config_file.add_device("fancy_device", Device(None, "2001:db8::42"))
```

In [ ]:
import time

from lemonbeat import *
from lemonbeat import config_file
from lemonbeat.value_properties import ValueProperties


def select_device(select_device):
    global mower
    mower = select_device


gw, devices = config_file.load_or_create()
widgets.interact(select_device, select_device=devices)

gw.include()
time.sleep(1)
assert gw.get_device_description()[DeviceDescriptionType.INCLUDED]

## Setup Mower

In [ ]:
import datetime
import IPython
from lemonbeat.mower_helpers import (decode_value_report, pack_date, Task, pack_schedule, StartingPoint,
                                     pack_starting_points, html_format_value_report)
from lemonbeat.mower_helpers import LonaControl, pack_lona_control


mower.include()
time.sleep(2)
assert mower.get_device_description()[DeviceDescriptionType.INCLUDED]

mower.set_status_level(Status.Level.DEBUG)
mower.set_timezone_offset(time.localtime().tm_gmtoff)
mower.save_config()
time.sleep(3)

mower_value_description = mower.get_value_description()
mower_values = ValueProperties(mower, value_description=mower_value_description)

## Values

In [ ]:
print(f"timezone offset: {mower.get_timezone_offset()}")

IPython.display.HTML(html_format_value_report(decode_value_report(mower.get_value(), mower_value_description)))

## Set Schedule

In [ ]:
mower_values.schedule_config = pack_schedule([
    Task(id=0, weekdays='0100000', start=datetime.time(13, 45), duration=datetime.timedelta(hours=1), a_id=0),
    Task(id=1, weekdays='1000000', start=datetime.time(13, 32), duration=datetime.timedelta(hours=1), a_id=0),
    Task(id=2, weekdays='1000000', start=datetime.time(17, 32), duration=datetime.timedelta(hours=1), a_id=0),
])

## Set Starting Points

In [ ]:
mower_values.starting_points = pack_starting_points([
    StartingPoint(loop_wire=2, distance=10, proportion=20, enabled=True, corridor_cut=False),
    StartingPoint(loop_wire=2, distance=20, proportion=5, enabled=True, corridor_cut=True),
    StartingPoint(loop_wire=2, distance=0, proportion=0, enabled=False, corridor_cut=False),
])

## TSS: Set Schedule Control Skip

In [ ]:
# schedule_state_control_skip
mower_values.schedule_state_control_skip = pack_schedule_states_control_skip([
    ScheduleStateControlSkip(id=0),
])

## Commands

In [ ]:
mower.include()

In [ ]:
mower.exclude()

In [ ]:
mower.get_device_description()

In [ ]:
mower.get_value_description()

In [ ]:
# measure rf link
mower_values.command = 7

In [ ]:
# trigger data update
mower_values.command = 21

In [ ]:
# reboot
mower_values.command = 31

In [ ]:
# CBTL: board reset
mower_values.command = 48

In [ ]:
# ntp force sync
mower_values.command = 32

In [ ]:
# park until next timer
mower_values.mower_timer = 0

In [ ]:
# park until further notice
mower_values.action_paused_until_1 = pack_date(datetime.datetime(2999, 1, 1, 0, 0))

In [ ]:
# resume schedule
mower_values.action_paused_until_1 = pack_date(datetime.datetime(1970, 1, 1, 0, 0))

In [ ]:
# legacy CBT2: start override timer (minutes)
mower_values.start_override_time = 2
mower_values.command = 8

In [ ]:
# legacy CBT2: resume schedule
mower_values.command = 9

In [ ]:
# legacy CBT2: park until further notice
mower_values.command = 10

In [ ]:
# legacy CBT2: park until next timer
mower_values.command = 11

In [ ]:
# legacy CBT2: set week timer
mower_values.command = 12

In [ ]:
# legacy CBT2: get week timer
mower_values.command = 18

In [ ]:
# legacy CBT2: trigger all values
mower_values.command = 30

In [ ]:
# CBTL: Enable All LONA Control Settings
mower_values.lona_control = pack_lona_control(LonaControl(
    global_enable=True, data_collection=True, debug_events=True,
    debug_data_collection=True, sensor_position_mapping=True))